# OMG 2차 일별 리텐션 분석

검증이 완료된 BigQuery 일별 마트에서 분석 대상 데이터를 불러와 다음 날 재방문을 분석한다.

- 시간 기준: UTC
- 물리 마트: `sns-analysis-prj.sns_analysis.mart_user_daily_state_analysis`
- 전체 관찰기간: 2023-07-24~2023-08-10
- 앵커 코호트: 2023-07-24~2023-07-26 중 가입일 이후 실제 활동이 1회 이상인 유저
- D+1 재방문: 오늘 활동한 유저가 다음 달력 날짜에도 `activity_flag = 1`인 경우


## 0. 환경 설정


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATASET_ID = "sns_analysis"
MART_TABLE = "mart_user_daily_state_analysis"
MART_FQN = f"{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}"

LOCATION = "asia-northeast3"
START_DATE = "2023-07-24"
INITIAL_END_DATE = "2023-07-26"
END_DATE = "2023-08-10"
MAX_BYTES = 1100 * 1024**2

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)


## 1. 물리 마트 조회

이 노트북에서는 마트를 다시 만들지 않는다. 기존 물리 마트에서 앵커 코호트와 분석에 필요한 열만 불러온다.


In [ ]:
load_sql = f"""
SELECT
    user_id,
    activity_date,
    week_start,
    analysis_phase,
    anchor_cohort_flag,
    signup_date,
    user_tenure_days,
    gender,
    group_id,
    school_id,
    grade,
    class_num,
    school_type,
    profile_match_flag,
    activity_flag,
    core_activity_flag,
    event_count,
    session_count,
    launch_app_count,
    question_start_count,
    question_complete_count,
    question_skip_count,
    received_question_open_count,
    question_tab_view_count,
    timeline_view_count,
    attendance_click_count,
    purchase_count,
    sent_vote_count,
    sent_unique_receiver_count,
    new_receiver_count,
    received_vote_count,
    received_unique_voter_count,
    new_voter_count,
    received_vote_concentration,
    reciprocal_partner_count_to_date,
    last_received_vote_date,
    days_since_last_received_vote,
    question_entry_flag,
    actual_vote_flag,
    question_entry_no_vote_flag,
    reward_open_flag,
    received_vote_flag,
    attendance_flag,
    class_activity_available_flag,
    user_behavior_state,
    class_active_user_count,
    active_classmates_daily
FROM `{MART_FQN}`
WHERE anchor_cohort_flag = 1
  AND activity_date BETWEEN DATE('{START_DATE}') AND DATE('{END_DATE}')
"""

dry_run_job = client.query(
    load_sql,
    job_config=bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False,
    ),
)

estimated_bytes = dry_run_job.total_bytes_processed
print(f"예상 처리량: {estimated_bytes / 1024**2:,.1f} MiB")

if estimated_bytes > MAX_BYTES:
    raise RuntimeError("예상 처리량이 MAX_BYTES를 초과했습니다.")


In [ ]:
daily_df = client.query(
    load_sql,
    job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=MAX_BYTES
    ),
).to_dataframe(create_bqstorage_client=False)

daily_df["activity_date"] = pd.to_datetime(daily_df["activity_date"])
daily_df["signup_date"] = pd.to_datetime(daily_df["signup_date"])

daily_df = daily_df.sort_values(
    ["user_id", "activity_date"]
).reset_index(drop=True)

print(f"daily_df: {len(daily_df):,}행")
print(f"유저 수: {daily_df['user_id'].nunique():,}명")
print(f"기간: {daily_df['activity_date'].min().date()} ~ {daily_df['activity_date'].max().date()}")


In [ ]:
daily_load_qa = pd.Series({
    "rows": len(daily_df),
    "users": daily_df["user_id"].nunique(),
    "min_date": daily_df["activity_date"].min().date(),
    "max_date": daily_df["activity_date"].max().date(),
    "duplicate_keys": daily_df.duplicated(
        ["user_id", "activity_date"]
    ).sum(),
    "negative_tenure_rows": (
        daily_df["user_tenure_days"] < 0
    ).sum(),
})

daily_load_qa


## 2. 분석 대상 확정

마트의 전체 날짜 구조 때문에 포함된 가입 전 유저-날짜 행을 제거한다. 이후 앵커 기간에 가입일 이후 실제 활동이 있었던 유저만 다시 선정한다. 가입일이 연결되지 않은 유저는 행동 분석에는 유지하고, 가입 경과일을 사용하는 세그먼트 분석에서 별도로 제외한다.


In [ ]:
valid_day_mask = (
    daily_df["user_tenure_days"].isna()
    | (daily_df["user_tenure_days"] >= 0)
)

initial_period_mask = daily_df["activity_date"].between(
    pd.Timestamp(START_DATE),
    pd.Timestamp(INITIAL_END_DATE),
)

valid_anchor_users = daily_df.loc[
    valid_day_mask
    & initial_period_mask
    & (daily_df["activity_flag"] == 1),
    "user_id",
].unique()

analysis_df = (
    daily_df.loc[
        valid_day_mask
        & daily_df["user_id"].isin(valid_anchor_users)
    ]
    .copy()
    .sort_values(["user_id", "activity_date"])
    .reset_index(drop=True)
)

# click_question_open의 의미가 드러나도록 분석용 별칭을 사용
analysis_df["received_question_open_flag"] = (
    analysis_df["reward_open_flag"]
)
analysis_df["analysis_state"] = analysis_df[
    "user_behavior_state"
].replace({"reward_opened": "received_question_opened"})


In [ ]:
analysis_qa = pd.Series({
    "rows": len(analysis_df),
    "users": analysis_df["user_id"].nunique(),
    "min_date": analysis_df["activity_date"].min().date(),
    "max_date": analysis_df["activity_date"].max().date(),
    "duplicate_keys": analysis_df.duplicated(
        ["user_id", "activity_date"]
    ).sum(),
    "negative_tenure_rows": (
        analysis_df["user_tenure_days"] < 0
    ).sum(),
})

analysis_qa


## 3. 다음 날 재방문 변수 생성

각 유저의 다음 행이 실제 다음 달력 날짜일 때만 결과를 사용한다. 2023-08-10은 다음 날 데이터가 없으므로 결과값을 결측치로 둔다.


In [ ]:
analysis_df["next_activity_date"] = (
    analysis_df.groupby("user_id")["activity_date"].shift(-1)
)
analysis_df["next_day_activity_flag"] = (
    analysis_df.groupby("user_id")["activity_flag"].shift(-1)
)

consecutive_day_mask = (
    analysis_df["next_activity_date"]
    == analysis_df["activity_date"] + pd.Timedelta(days=1)
)

analysis_df.loc[
    ~consecutive_day_mask,
    "next_day_activity_flag",
] = pd.NA

analysis_df["next_day_activity_flag"] = (
    analysis_df["next_day_activity_flag"].astype("Int64")
)


In [ ]:
transition_qa = pd.Series({
    "next_day_observed_rows": analysis_df[
        "next_day_activity_flag"
    ].notna().sum(),
    "last_or_unobserved_rows": analysis_df[
        "next_day_activity_flag"
    ].isna().sum(),
    "nonconsecutive_middle_rows": (
        analysis_df["next_activity_date"].notna()
        & ~consecutive_day_mask
    ).sum(),
})

transition_qa


## 4. 전체 및 날짜별 D+1 재방문율

분모는 다음 날 관찰이 가능하면서 오늘 `activity_flag = 1`인 유저-날짜이다. 동일 유저가 여러 날짜에 포함될 수 있으므로 이 단계의 단위는 고유 유저 수가 아니라 활성 유저-날짜이다.


In [ ]:
active_day_df = analysis_df.loc[
    (analysis_df["activity_flag"] == 1)
    & analysis_df["next_day_activity_flag"].notna()
].copy()

overall_retention = pd.Series({
    "active_user_days": len(active_day_df),
    "next_day_returners": active_day_df[
        "next_day_activity_flag"
    ].sum(),
    "next_day_retention_pct": (
        active_day_df["next_day_activity_flag"].mean() * 100
    ),
})

overall_retention


In [ ]:
daily_retention = (
    active_day_df
    .groupby("activity_date", as_index=False)
    .agg(
        active_users=("user_id", "nunique"),
        next_day_returners=("next_day_activity_flag", "sum"),
    )
)

daily_retention["next_day_retention_pct"] = (
    daily_retention["next_day_returners"]
    / daily_retention["active_users"]
    * 100
)

daily_retention


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(
    daily_retention["activity_date"],
    daily_retention["next_day_retention_pct"],
    marker="o",
)
plt.title("날짜별 D+1 재방문율")
plt.xlabel("")
plt.ylabel("재방문율(%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. 다음 분석

전체 기준선을 확인한 뒤 아래 순서로 진행한다.

1. 행동 상태별 D+1 재방문율
2. 일별 상태 전이행렬
3. 질문 진입·실제 투표·투표 수신·받은 질문 열람 비교
4. 신규 관계 및 사회적 보상 감소와 이탈
5. 활동량·가입 경과일을 통제한 학급 환경 비교
6. 투자 우선순위 및 실험안 도출
